# Proper Train/Test Split - Preventing Data Leakage

**Educational Purpose**: Learn why and how to prevent data leakage in time series models

## The Problem: Data Leakage

**Bad Practice #1 - Random Split (Common Mistake)**:
```python
# ❌ WRONG: Random split mixes past and future
data = load_data("2025-11-04 to 2025-12-04")
train, test = train_test_split(data, test_size=0.2)  # Random!
model.fit(train)
results = model.evaluate(test)  # Includes data from training period!
```

**Bad Practice #2 - Features Before Split (CRITICAL!)**:
```python
# ❌ WRONG: Features computed on ALL data before split
features = generate_all_features(data)  # RSI, MACD, etc. on ALL data
train, test = temporal_split(features)  # Too late! Features already leaked
```

**Why This Is Wrong**:
- Technical indicators (RSI, MACD, SMA) use rolling windows
- Computing them on all data means test features include future information
- Results are 5-15% more optimistic than reality
- Model WILL fail in production

## The Solution: Split FIRST, Then Feature Engineering

**Good Practice (StatefulFeatureEngineer)**:
```python
# ✓ CORRECT: Split data FIRST, then generate features separately
train_df, test_df = temporal_split(raw_data)  # Split raw OHLCV first!

engineer = StatefulFeatureEngineer()
train_features = engineer.fit_transform(train_df)   # Fit on train only
test_features = engineer.transform(test_df)         # Transform using train stats
```

This notebook demonstrates **three methods**:
1. **Quick Split with StatefulFeatureEngineer** - Fast, correct for Phase 1
2. **Walk-Forward Validation** - Gold standard, most realistic
3. **Financial Metrics** - Evaluate with transaction costs

**Expected Runtime**: ~5-10 minutes

In [ ]:
"""Import required libraries."""
from __future__ import annotations

import asyncio
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from loguru import logger
from plotly.subplots import make_subplots
from rich.console import Console
from rich.table import Table

from price_stradamus.config.constants import (
    FEATURE_WARMUP_PERIOD,
    REALISTIC_DIR_ACC_MAX,
    SUSPICIOUS_DIR_ACC,
)
from price_stradamus.config.settings import settings
from price_stradamus.data.database import DatabaseManager
from price_stradamus.data.preprocessor import DataPreprocessor
# NEW: Use StatefulFeatureEngineer instead of deprecated FeatureEngineer
from price_stradamus.data.stateful_features import StatefulFeatureEngineer
from price_stradamus.evaluation.metrics import MetricsCalculator
from price_stradamus.evaluation.financial_metrics import (
    FinancialMetricsCalculator,
    TradingCosts,
)
from price_stradamus.models.registry import ModelRegistry

console = Console()

# Configuration
CONFIG = {
    "model": "nbeats",
    "symbol": "BTCUSDT",
    "timeframe": "1m",
    "days": 30,
    "input_chunk_length": 60,
    "output_chunk_length": 5,
    "n_epochs": 50,
}

# Trading costs for realistic evaluation
COSTS = TradingCosts(commission_pct=0.001, slippage_pct=0.0005)

console.print("[bold cyan]Proper Train/Test Split Demo[/bold cyan]")
console.print(f"Model: {CONFIG['model']}")
console.print(f"Data: {CONFIG['symbol']} {CONFIG['timeframe']}")
console.print(f"Transaction costs: {COSTS.round_trip_cost*100:.2f}% per round trip")

In [ ]:
"""Load Bitcoin data from database."""

db = DatabaseManager(settings.database_url)
await db.initialize()

end_date = datetime.now()
start_date = end_date - timedelta(days=CONFIG["days"])

df = await db.get_ohlcv(
    symbol=CONFIG["symbol"],
    timeframe=CONFIG["timeframe"],
    start_date=start_date,
    end_date=end_date,
)

await db.close()

data_start = df["timestamp"].min()
data_end = df["timestamp"].max()

console.print(f"[green]✓ Loaded {len(df)} candles[/green]")
console.print(f"  Data range: {data_start} to {data_end}")

df.head()

In [ ]:
"""Preprocess data - DO NOT generate features yet!

CRITICAL: Features must be generated AFTER the train/test split.
We only do basic preprocessing here (validation, missing values).
"""

preprocessor = DataPreprocessor()
df_clean = preprocessor.validate_ohlcv(df)
df_clean = preprocessor.handle_missing_values(df_clean)

# Ensure timestamp is datetime for splitting
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"], utc=True)

console.print(f"[green]✓ Preprocessed {len(df_clean)} rows[/green]")
console.print(f"[yellow]⚠️  Features NOT generated yet - will generate AFTER split[/yellow]")

## Method 1: Simple Temporal Split with StatefulFeatureEngineer

**The Correct Pattern**:
1. Split raw OHLCV data temporally (70/15/15)
2. Create StatefulFeatureEngineer
3. `fit_transform()` on training data - learns statistics
4. `transform()` on validation/test data - uses training statistics

**Why StatefulFeatureEngineer?**
- Remembers normalization statistics from training data
- Applies same transformation to test data
- Prevents feature leakage from looking ahead
- Drops warmup period (first ~200 rows with incomplete indicators)

**Pros**:
- Fast to implement
- Correct (no data leakage!)
- Sufficient for initial model evaluation

**Cons**:
- Single test period (may not be representative)
- Doesn't test model stability over time

In [ ]:
"""Method 1: Correct temporal split with StatefulFeatureEngineer.

CRITICAL ORDER:
1. Split raw OHLCV data FIRST
2. Generate features on each split SEPARATELY
"""

# Step 1: Split RAW data (before features!)
train_size = int(len(df_clean) * 0.70)
val_size = int(len(df_clean) * 0.15)

train_df = df_clean.iloc[:train_size].copy()
val_df = df_clean.iloc[train_size:train_size + val_size].copy()
test_df = df_clean.iloc[train_size + val_size:].copy()

console.print("\n[bold yellow]Method 1: Correct Split with StatefulFeatureEngineer[/bold yellow]\n")
console.print("[green]Step 1: Split RAW OHLCV data (before features!)[/green]")

table = Table(title="Data Splits (70/15/15) - RAW OHLCV")
table.add_column("Set", style="cyan")
table.add_column("Size", style="yellow")
table.add_column("Start", style="green")
table.add_column("End", style="green")
table.add_column("%", style="magenta")

table.add_row(
    "Training",
    str(len(train_df)),
    str(train_df["timestamp"].iloc[0]),
    str(train_df["timestamp"].iloc[-1]),
    "70%",
)
table.add_row(
    "Validation",
    str(len(val_df)),
    str(val_df["timestamp"].iloc[0]),
    str(val_df["timestamp"].iloc[-1]),
    "15%",
)
table.add_row(
    "Test",
    str(len(test_df)),
    str(test_df["timestamp"].iloc[0]),
    str(test_df["timestamp"].iloc[-1]),
    "15%",
)

console.print(table)

# Step 2: Generate features AFTER split using StatefulFeatureEngineer
console.print("\n[green]Step 2: Generate features (fit on train, transform val/test)[/green]")

engineer = StatefulFeatureEngineer(warmup_period=FEATURE_WARMUP_PERIOD)

# fit_transform on training data - learns statistics
train_features = engineer.fit_transform(train_df)

# transform on val/test - uses training statistics (NO leakage!)
val_features = engineer.transform(val_df, drop_warmup=False)
test_features = engineer.transform(test_df, drop_warmup=False)

console.print(f"[green]✓ Generated {len(engineer.feature_list)} features[/green]")
console.print(f"  Train: {len(train_features)} samples (after warmup drop)")
console.print(f"  Val:   {len(val_features)} samples")
console.print(f"  Test:  {len(test_features)} samples")

# Convert to Darts TimeSeries
train_ts = engineer.to_darts_timeseries(train_features, value_cols=["close"])
val_ts = engineer.to_darts_timeseries(val_features, value_cols=["close"])
test_ts = engineer.to_darts_timeseries(test_features, value_cols=["close"])

In [ ]:
"""Visualize the temporal split boundaries."""

# Create timeline visualization
fig = go.Figure()

# Training period
fig.add_trace(go.Scatter(
    x=[train_ts.time_index[0], train_ts.time_index[-1]],
    y=[1, 1],
    mode='lines+markers',
    name='Training',
    line=dict(color='green', width=20),
    marker=dict(size=15, symbol='square'),
))

# Validation period
fig.add_trace(go.Scatter(
    x=[val_ts.time_index[0], val_ts.time_index[-1]],
    y=[1, 1],
    mode='lines+markers',
    name='Validation',
    line=dict(color='yellow', width=20),
    marker=dict(size=15, symbol='square'),
))

# Test period
fig.add_trace(go.Scatter(
    x=[test_ts.time_index[0], test_ts.time_index[-1]],
    y=[1, 1],
    mode='lines+markers',
    name='Test',
    line=dict(color='red', width=20),
    marker=dict(size=15, symbol='square'),
))

fig.update_layout(
    title="Temporal Data Split Visualization",
    xaxis_title="Date",
    yaxis=dict(visible=False),
    template="plotly_dark",
    height=300,
    showlegend=True,
    legend=dict(orientation="h", y=1.1, x=0.5, xanchor="center"),
)

fig.show()

console.print("[green]✓ Timeline shows no overlap between sets[/green]")

In [ ]:
"""CRITICAL: Verify no data leakage in splits."""

console.print("\n[bold red]🔍 CHECKING FOR DATA LEAKAGE...[/bold red]\n")

def check_temporal_leakage(earlier_ts, later_ts):
    """Check if later time series starts after earlier one ends."""
    earlier_end = earlier_ts.time_index[-1]
    later_start = later_ts.time_index[0]
    has_leakage = later_start <= earlier_end
    gap_hours = (later_start - earlier_end).total_seconds() / 3600
    return {"has_leakage": has_leakage, "gap": gap_hours}

# Check train vs validation
result_val = check_temporal_leakage(train_ts, val_ts)
if result_val["has_leakage"]:
    console.print("[red]✗ LEAKAGE DETECTED: Train/Val overlap![/red]")
    raise ValueError("Data leakage in train/val split")
else:
    console.print("[green]✓ Train/Val: No temporal leakage[/green]")
    console.print(f"  Gap: {result_val['gap']:.1f} hours")

# Check validation vs test
result_test = check_temporal_leakage(val_ts, test_ts)
if result_test["has_leakage"]:
    console.print("[red]✗ LEAKAGE DETECTED: Val/Test overlap![/red]")
    raise ValueError("Data leakage in val/test split")
else:
    console.print("[green]✓ Val/Test: No temporal leakage[/green]")
    console.print(f"  Gap: {result_test['gap']:.1f} hours")

# Verify feature statistics were fit on training only
console.print(f"\n[green]✓ Feature statistics fit on training data only[/green]")
console.print(f"  Normalization stats computed from {len(train_features)} training samples")

console.print("\n[bold green]✓✓✓ ALL CHECKS PASSED - NO DATA LEAKAGE ✓✓✓[/bold green]")

In [ ]:
"""Train model using simple split."""

console.print("\n[cyan]Training model on training set only...[/cyan]")

# Create model
model = ModelRegistry.create_model(
    CONFIG["model"],
    input_chunk_length=CONFIG["input_chunk_length"],
    output_chunk_length=CONFIG["output_chunk_length"],
    n_epochs=CONFIG["n_epochs"],
)

# Train (only on training data!)
model.fit(train_ts, val_ts)

console.print("[green]✓ Training complete[/green]")

In [ ]:
"""Evaluate model on unseen test data with financial metrics."""

console.print("\n[cyan]Evaluating on test set (model has NEVER seen this data)...[/cyan]")

# Predict
predictions = model.predict(n=len(test_ts), series=train_ts)

# Calculate metrics
actuals = test_ts.values().flatten()
preds = predictions.values().flatten()[:len(actuals)]

# Calculate all metrics including financial
metrics = MetricsCalculator.calculate_all_with_financial(
    actuals, preds, include_financial=True, trading_costs=COSTS
)

# Display results
console.print("\n[bold green]Test Set Results (NO DATA LEAKAGE!):[/bold green]")

table = Table(title="Error Metrics")
table.add_column("Metric", style="cyan")
table.add_column("Value", style="green")

table.add_row("MAE", f"{metrics['mae']:.2f} USDT")
table.add_row("RMSE", f"{metrics['rmse']:.2f} USDT")
table.add_row("MAPE", f"{metrics['mape']:.2f}%")
table.add_row("Directional Accuracy", f"{metrics['directional_accuracy']:.2f}%")
table.add_row("R²", f"{metrics['r2']:.4f}")

console.print(table)

# Financial metrics
console.print("\n[bold]Financial Metrics (with transaction costs):[/bold]")

fin_table = Table(title=f"Trading Performance (Costs: {COSTS.round_trip_cost*100:.2f}% per trade)")
fin_table.add_column("Metric", style="cyan")
fin_table.add_column("Value", style="green")

fin_table.add_row("Gross P&L", f"${metrics['fin_gross_pnl']:.2f}")
fin_table.add_row("Total Costs", f"${metrics['fin_total_costs']:.2f}")
fin_table.add_row("Net P&L", f"${metrics['fin_net_pnl']:.2f}")
fin_table.add_row("Sharpe Ratio", f"{metrics['fin_sharpe_ratio']:.2f}")
fin_table.add_row("Win Rate", f"{metrics['fin_win_rate']:.1f}%")
fin_table.add_row("Max Drawdown", f"{metrics['fin_max_drawdown_pct']:.1f}%")

console.print(fin_table)

# Check for suspicious results
if metrics['directional_accuracy'] > SUSPICIOUS_DIR_ACC:
    console.print(f"\n[bold red]⚠️  WARNING: Accuracy {metrics['directional_accuracy']:.1f}% is suspiciously high![/bold red]")
    console.print("[red]  Realistic BTC 1-min accuracy is 51-55%. Check for bugs.[/red]")
elif metrics['directional_accuracy'] > REALISTIC_DIR_ACC_MAX:
    console.print(f"\n[yellow]⚠️  Accuracy {metrics['directional_accuracy']:.1f}% is above typical range (51-55%)[/yellow]")

In [ ]:
"""Plot predictions vs actuals for simple split."""

fig = go.Figure()

# Actual prices
fig.add_trace(go.Scatter(
    x=test_ts.time_index,
    y=actuals,
    mode='lines+markers',
    name='Actual',
    line=dict(color='white', width=3),
))

# Predicted prices
fig.add_trace(go.Scatter(
    x=test_ts.time_index,
    y=preds,
    mode='lines+markers',
    name='Predicted',
    line=dict(color='cyan', width=2, dash='dash'),
))

fig.update_layout(
    title=f"Simple Split Results - {CONFIG['model']} (RMSE: {metrics['rmse']:.2f} USDT)",
    xaxis_title="Time",
    yaxis_title="Price (USDT)",
    template="plotly_dark",
    height=600,
    hovermode='x unified',
)

fig.show()

## Method 2: Walk-Forward Validation (Gold Standard)

**When to use**: Production evaluation, research papers, final model selection

**How it works with StatefulFeatureEngineer**:
1. For each fold:
   - Split data temporally (train window → test window)
   - Create NEW StatefulFeatureEngineer
   - `fit_transform()` on training data
   - `transform()` on test data
   - Train model and evaluate
2. Aggregate results across all folds

**Why regenerate features per fold?**
- Each fold has different training data
- Feature normalization stats should be from that fold's training data
- Prevents any cross-fold leakage

**Pros**:
- Tests model on multiple time periods
- Simulates real-world deployment
- Provides robust performance estimates with confidence intervals
- Reveals model stability over time

**Cons**:
- Slower (retrains model AND features on each fold)
- More complex to implement
- Requires more historical data

In [ ]:
"""Method 2: Walk-forward validation setup with StatefulFeatureEngineer."""

console.print("\n[bold yellow]Method 2: Walk-Forward Validation[/bold yellow]\n")

# Walk-forward parameters
TRAIN_WINDOW_DAYS = 20  # 20 days of training data per fold
TEST_WINDOW_DAYS = 5    # 5 days of test data per fold
SAMPLES_PER_DAY = 24 * 60  # 1440 for 1-minute data

train_samples = TRAIN_WINDOW_DAYS * SAMPLES_PER_DAY
test_samples = TEST_WINDOW_DAYS * SAMPLES_PER_DAY

# Calculate number of possible folds
total_data = len(df_clean)
available_for_testing = total_data - train_samples
num_folds = max(1, available_for_testing // test_samples)

console.print(f"[green]Walk-forward configuration:[/green]")
console.print(f"  Train window: {TRAIN_WINDOW_DAYS} days ({train_samples} samples)")
console.print(f"  Test window: {TEST_WINDOW_DAYS} days ({test_samples} samples)")
console.print(f"  Number of folds: {num_folds}")

# Display first 3 windows
table = Table(title="Walk-Forward Windows (First 3)")
table.add_column("Fold", style="cyan")
table.add_column("Train Range", style="green")
table.add_column("Test Range", style="yellow")

for i in range(min(3, num_folds)):
    test_end_idx = total_data - (num_folds - i - 1) * test_samples
    test_start_idx = test_end_idx - test_samples
    train_end_idx = test_start_idx
    train_start_idx = max(0, train_end_idx - train_samples)
    
    train_start = df_clean.iloc[train_start_idx]["timestamp"]
    train_end = df_clean.iloc[train_end_idx-1]["timestamp"]
    test_start = df_clean.iloc[test_start_idx]["timestamp"]
    test_end = df_clean.iloc[test_end_idx-1]["timestamp"]
    
    table.add_row(
        str(i+1),
        f"{str(train_start)[:16]} → {str(train_end)[:16]}",
        f"{str(test_start)[:16]} → {str(test_end)[:16]}",
    )

console.print(table)

if num_folds > 3:
    console.print(f"[dim]... and {num_folds - 3} more folds[/dim]")

In [ ]:
"""Run walk-forward validation with StatefulFeatureEngineer (this takes time!)."""

from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn

console.print("\n[cyan]Running walk-forward validation (regenerating features per fold)...[/cyan]")

wf_results = []

with Progress(
    SpinnerColumn(),
    TextColumn("[progress.description]{task.description}"),
    BarColumn(),
    console=console,
) as progress:
    
    task = progress.add_task(
        "[cyan]Walk-forward...",
        total=num_folds,
    )
    
    for fold in range(num_folds):
        progress.update(task, description=f"[cyan]Fold {fold+1}/{num_folds}...")
        
        # Calculate fold boundaries
        test_end_idx = total_data - (num_folds - fold - 1) * test_samples
        test_start_idx = test_end_idx - test_samples
        train_end_idx = test_start_idx
        train_start_idx = max(0, train_end_idx - train_samples)
        
        # Split RAW data for this fold
        fold_train_df = df_clean.iloc[train_start_idx:train_end_idx].copy()
        fold_test_df = df_clean.iloc[test_start_idx:test_end_idx].copy()
        
        # CRITICAL: Create NEW StatefulFeatureEngineer for each fold
        fold_engineer = StatefulFeatureEngineer(warmup_period=FEATURE_WARMUP_PERIOD)
        
        # fit_transform on this fold's training data
        fold_train_features = fold_engineer.fit_transform(fold_train_df)
        # transform test using this fold's training statistics
        fold_test_features = fold_engineer.transform(fold_test_df, drop_warmup=False)
        
        # Convert to time series
        fold_train_ts = fold_engineer.to_darts_timeseries(fold_train_features, value_cols=["close"])
        fold_test_ts = fold_engineer.to_darts_timeseries(fold_test_features, value_cols=["close"])
        
        # Train model for this fold
        fold_model = ModelRegistry.create_model(
            CONFIG["model"],
            input_chunk_length=CONFIG["input_chunk_length"],
            output_chunk_length=CONFIG["output_chunk_length"],
            n_epochs=CONFIG["n_epochs"],
        )
        
        fold_model.fit(fold_train_ts)
        
        # Predict and evaluate
        fold_pred = fold_model.predict(n=len(fold_test_ts), series=fold_train_ts)
        
        fold_actuals = fold_test_ts.values().flatten()
        fold_preds = fold_pred.values().flatten()[:len(fold_actuals)]
        
        # Calculate metrics including financial
        fold_metrics = MetricsCalculator.calculate_all_with_financial(
            fold_actuals, fold_preds, include_financial=True, trading_costs=COSTS
        )
        fold_metrics["fold"] = fold + 1
        wf_results.append(fold_metrics)
        
        progress.advance(task)

console.print(f"\n[green]✓ Completed {len(wf_results)} folds with proper feature regeneration[/green]")

In [ ]:
"""Calculate aggregate statistics from walk-forward validation."""

# Aggregate metrics
metrics_to_aggregate = [
    ("mae", "MAE", "{:.2f}"),
    ("rmse", "RMSE", "{:.2f}"),
    ("mape", "MAPE %", "{:.2f}"),
    ("directional_accuracy", "Dir Accuracy %", "{:.2f}"),
    ("fin_net_pnl", "Net P&L $", "{:.2f}"),
    ("fin_sharpe_ratio", "Sharpe Ratio", "{:.2f}"),
    ("fin_win_rate", "Win Rate %", "{:.1f}"),
]

# Calculate stats
avg_metrics = {}
std_metrics = {}

for key, _, _ in metrics_to_aggregate:
    values = [m[key] for m in wf_results]
    avg_metrics[key] = np.mean(values)
    std_metrics[key] = np.std(values)

# Display results
console.print("\n[bold green]Walk-Forward Results (Average Across Folds):[/bold green]")

table = Table(title=f"Metrics ({len(wf_results)} folds, costs: {COSTS.round_trip_cost*100:.2f}%)")
table.add_column("Metric", style="cyan")
table.add_column("Average", style="green")
table.add_column("Std Dev", style="yellow")

for key, name, fmt in metrics_to_aggregate:
    table.add_row(
        name,
        fmt.format(avg_metrics[key]),
        f"±{fmt.format(std_metrics[key])}",
    )

console.print(table)

# Stability assessment
rmse_cv = (std_metrics["rmse"] / avg_metrics["rmse"]) * 100 if avg_metrics["rmse"] > 0 else 0

console.print(f"\n[bold]Model Stability:[/bold]")
console.print(f"  RMSE Coefficient of Variation: {rmse_cv:.1f}%")
if rmse_cv < 20:
    console.print(f"  [green]✓ Excellent stability (CV < 20%)[/green]")
elif rmse_cv < 30:
    console.print(f"  [yellow]⚠ Moderate stability (CV 20-30%)[/yellow]")
else:
    console.print(f"  [red]✗ Poor stability (CV > 30%) - investigate further[/red]")

console.print(f"\n[green]✓ Evaluated on {len(wf_results)} folds with NO DATA LEAKAGE[/green]")

In [ ]:
"""Compare simple split vs walk-forward validation."""

console.print("\n" + "="*70)
console.print(" COMPARISON: Simple Split vs Walk-Forward Validation ")
console.print("="*70)

table = Table()
table.add_column("Method", style="cyan")
table.add_column("RMSE", style="yellow")
table.add_column("Dir Acc %", style="green")
table.add_column("Sharpe", style="magenta")
table.add_column("Evaluation", style="dim")

table.add_row(
    "Simple Split (70/15/15)",
    f"{metrics['rmse']:.2f}",
    f"{metrics['directional_accuracy']:.2f}",
    f"{metrics['fin_sharpe_ratio']:.2f}",
    "Single test period",
)

table.add_row(
    "Walk-Forward (Gold Standard)",
    f"{avg_metrics['rmse']:.2f} ±{std_metrics['rmse']:.2f}",
    f"{avg_metrics['directional_accuracy']:.2f} ±{std_metrics['directional_accuracy']:.2f}",
    f"{avg_metrics['fin_sharpe_ratio']:.2f} ±{std_metrics['fin_sharpe_ratio']:.2f}",
    f"{len(wf_results)} independent folds",
)

console.print(table)

# Interpretation
console.print(f"\n[bold]KEY INSIGHTS:[/bold]")

rmse_diff = abs(metrics['rmse'] - avg_metrics['rmse'])
if rmse_diff < metrics['rmse'] * 0.1:  # Within 10%
    console.print(f"  [green]✓ Both methods agree (within 10%) - model is stable[/green]")
else:
    console.print(f"  [yellow]⚠ Methods differ by {rmse_diff:.2f} RMSE - investigate further[/yellow]")

if avg_metrics['directional_accuracy'] > SUSPICIOUS_DIR_ACC:
    console.print(f"  [red]✗ Accuracy too high ({avg_metrics['directional_accuracy']:.1f}%) - possible bug[/red]")
elif avg_metrics['directional_accuracy'] > REALISTIC_DIR_ACC_MAX:
    console.print(f"  [yellow]⚠ Accuracy above typical range (51-55%)[/yellow]")
else:
    console.print(f"  [green]✓ Accuracy in realistic range for BTC 1-min[/green]")

console.print(f"\n[bold]RECOMMENDATIONS:[/bold]")
console.print(f"  → Use simple split for quick experiments")
console.print(f"  → Use walk-forward for final model evaluation")
console.print(f"  → Report walk-forward results with standard deviations")
console.print(f"  → Include transaction costs in all evaluations")

console.print("="*70)

## Summary: Preventing Data Leakage

### Key Takeaways

1. **Split RAW data first, THEN generate features** - This is the most important rule!
2. **Use StatefulFeatureEngineer** - It implements the correct fit-transform pattern
3. **Always use temporal splits** for time series data (never random)
4. **Walk-forward is the gold standard** for final evaluation
5. **Include transaction costs** to get realistic P&L estimates

### The Correct Pattern

```python
# ✅ CORRECT: Split first, then features
from price_stradamus.data.stateful_features import StatefulFeatureEngineer

# 1. Split RAW OHLCV data
train_df, test_df = temporal_split(raw_ohlcv_data)

# 2. Create feature engineer
engineer = StatefulFeatureEngineer(warmup_period=200)

# 3. Fit on train (learns normalization stats)
train_features = engineer.fit_transform(train_df)

# 4. Transform test (uses train stats - no leakage!)
test_features = engineer.transform(test_df)

# 5. Train and evaluate
model.fit(train_features)
predictions = model.predict(test_features)

# 6. Evaluate with costs
metrics = MetricsCalculator.calculate_all_with_financial(
    actuals, predictions, trading_costs=TradingCosts()
)
```

### What You Learned

- ✓ Why generating features before split causes data leakage
- ✓ How to use StatefulFeatureEngineer correctly
- ✓ How to implement walk-forward validation
- ✓ How to include transaction costs in evaluation
- ✓ What realistic BTC 1-min accuracy looks like (51-55%)

### Next Steps

- Apply these techniques to your own models
- Use the CLI: `price-stradamus train --model nbeats` (uses walk-forward by default!)
- See [backtest_visualization.ipynb](../03_results/backtest_visualization.ipynb) for visualization

### References

- [Time Series Cross-Validation (scikit-learn)](https://scikit-learn.org/stable/modules/cross_validation.html#time-series-split)
- [Avoiding Look-Ahead Bias](https://www.quantstart.com/articles/Avoiding-Look-Ahead-Bias/)